# 01 — Frame Extraction → Lance `frames` Table

**Purpose:** Use Ray to distribute GPU-accelerated frame extraction across workers, writing the results directly to a Lance dataset stored in a Databricks Volume.

**What this notebook does:**

1. **Configure paths and schema.** Define the UC Volume path where raw videos live and where the Lance `frames` dataset will be written. Declare the PyArrow schema for the frames table upfront so all workers write consistent batches.

2. **Initialize Ray on the cluster.** Start a Ray cluster using the Databricks Ray integration. Each worker will be assigned a GPU for frame decoding.

3. **Extract frames per video (Ray worker).** Each worker receives a list of video paths. For each video, it uses `decord` (GPU-accelerated) or `ffmpeg` to decode frames at a configurable sampling rate (e.g., 1 fps, every N frames, or keyframes only). Each decoded frame is JPEG-encoded as bytes and bundled with its metadata (`frame_id`, `video_id`, `frame_number`, `timestamp_ms`, `width`, `height`, `extracted_at`) into a PyArrow `RecordBatch`.

4. **Write inline binary to Lance.** Workers call `lance.write_dataset(batch, frames_path, mode='append', schema=schema)` directly — no Spark intermediate layer. Inline JPEG bytes (rather than file path references) are essential here: Lance's O(1) random-access guarantee only applies when the data being accessed lives in the Lance file itself. A path reference would require a second I/O hop to Volumes storage on every training sample fetch.

5. **Verify the dataset.** After all workers complete, open the Lance dataset on the driver and confirm row count, schema, and that the `image` column contains non-null bytes. Demonstrate random-access reads: `ds.take([0, 1000, 50000])` should return instantly regardless of dataset size.

**Inputs:** Raw video files in a Databricks Volume (`/Volumes/{catalog}/{schema}/{volume}/videos/`)

**Outputs:** Lance `frames` dataset at `/Volumes/{catalog}/{schema}/{volume}/frames/` — version 1, containing `image` + metadata columns, no embeddings yet.

**Next:** Run `02_feature_engineering.ipynb` to compute CLIP embeddings and add them as a new dataset version.